In [ ]:
# Importing libraries
import numpy as np
import pandas as pd

# Cleaning `googleplaystore.csv` (Apps)

In [2]:
# Loading the raw apps data
apps_raw_df = pd.read_csv("googleplaystore.csv")

print("Shape:", apps_raw_df.shape)
apps_raw_df.head()

Shape: (10841, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [3]:
# Checking data types
# We noticed Reviews, Installs and Price are object, even though they represent numbers.
apps_raw_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [4]:
# Checking for null values per column
apps_raw_df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [5]:
# Checking for fully duplicated rows
print("Duplicate rows:", apps_raw_df.duplicated().sum())

Duplicate rows: 483


In [ ]:
# Findings
    # `Rating` has missing values. We will be filling rather than dropping.
    # `Type` and `Content Rating` each have only 1 missing value. We will drop.
    # `Current Ver` and `Android Ver` have a few missing values We will fill them with `"Unknown".
    # `Reviews`, `Installs`, and `Price` are stored as **text**, not numbers, because of characters like `,`, `+`, and `$` mixed in.
    # There are duplicate rows to remove.

In [6]:
# Creating the copy datadrame
apps_clean_df = apps_raw_df.copy()

# Removing duplicated rows
apps_clean_df = apps_clean_df.drop_duplicates()

print("Shape after removing duplicates:", apps_clean_df.shape)

Shape after removing duplicates: (10358, 13)


In [7]:
# Filling Rating nulls with the column mean
rating_mean = apps_clean_df["Rating"].mean()
apps_clean_df["Rating"] = apps_clean_df["Rating"].fillna(rating_mean)

print(f"Mean rating used to fill nulls: {rating_mean:.2f}")

Mean rating used to fill nulls: 4.19


In [ ]:
# Droping the rows where Type or Content Rating is missing
apps_clean_df = apps_clean_df.dropna(subset=["Type", "Content Rating"])

print("Shape after dropping Type/Content Rating nulls:", apps_clean_df.shape)

Shape after dropping Type/Content Rating nulls: (10356, 13)


In [9]:
# Filling remaining nulls with "Unknown"
apps_clean_df["Current Ver"] = apps_clean_df["Current Ver"].fillna("Unknown")
apps_clean_df["Android Ver"] = apps_clean_df["Android Ver"].fillna("Unknown")

# Confirming there are no nulls left
apps_clean_df.isnull().sum()

App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       0
Android Ver       0
dtype: int64

In [10]:
# Converting Reviews column into int
apps_clean_df["Reviews"] = apps_clean_df["Reviews"].astype(int)

In [ ]:
# Removing the comma and the plus sign from Installs column and then convert into int
apps_clean_df["Installs"] = apps_clean_df["Installs"].str.replace(",", "", regex=False)
apps_clean_df["Installs"] = apps_clean_df["Installs"].str.replace("+", "", regex=False)
apps_clean_df["Installs"] = apps_clean_df["Installs"].astype(int)

In [12]:
# Price: values look like "$4.99" or "0" -- remove the dollar sign, then convert
apps_clean_df["Price"] = apps_clean_df["Price"].str.replace("$", "", regex=False)
apps_clean_df["Price"] = apps_clean_df["Price"].astype(float)

# Check the fix worked
apps_clean_df[["Reviews", "Installs", "Price"]].dtypes

Reviews       int64
Installs      int64
Price       float64
dtype: object

## 5. Build the `categories` lookup table

Remember: `Category` has **one value per app** (e.g., `'ART_AND_DESIGN'`) — a clean one-to-many relationship, exactly like `ticket_class` in the Titanic example. We turn it into its own lookup table with an ID, then replace the text column in `apps` with a foreign key.

Steps:
1. Get the list of unique category names.
2. Build a small DataFrame: `category_id`, `category_name`.
3. Build a dictionary mapping `category_name → category_id`.
4. Use `.map()` with that dictionary to create the `category_id` column in the apps table.

In [13]:
# Step 1 & 2: unique categories -> lookup table
unique_categories = sorted(apps_clean_df["Category"].unique())

categories_df = pd.DataFrame({
    "category_id": range(1, len(unique_categories) + 1),
    "category_name": unique_categories
})

print("Number of unique categories:", len(categories_df))
categories_df.head()

Number of unique categories: 33


,category_id,category_name
0,1,ART_AND_DESIGN
1,2,AUTO_AND_VEHICLES
2,3,BEAUTY
3,4,BOOKS_AND_REFERENCE
4,5,BUSINESS


In [14]:
# Step 3 & 4: map category_name -> category_id using a dictionary
category_to_id = dict(zip(categories_df["category_name"], categories_df["category_id"]))

apps_clean_df["category_id"] = apps_clean_df["Category"].map(category_to_id)

apps_clean_df[["App", "Category", "category_id"]].head()

,App,Category,category_id
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,1
1,Coloring book moana,ART_AND_DESIGN,1
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,1
3,Sketch - Draw & Paint,ART_AND_DESIGN,1
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,1


## 6. Create the `app_id` primary key

We reset the index and use it to generate a clean, sequential primary key for the `apps` table.

In [15]:
# Reset the index after all the dropna() calls (the old index has gaps now)
apps_clean_df = apps_clean_df.reset_index(drop=True)

# Create app_id starting at 1
apps_clean_df["app_id"] = apps_clean_df.index + 1

apps_clean_df[["app_id", "App", "category_id"]].head()

,app_id,App,category_id
0,1,Photo Editor & Candy Camera & Grid & ScrapBook,1
1,2,Coloring book moana,1
2,3,"U Launcher Lite – FREE Live Cool Themes, Hide ...",1
3,4,Sketch - Draw & Paint,1
4,5,Pixel Draw - Number Art Coloring Book,1


## 7. Build the final `apps` table

Select only the columns we need, in the order matching our database schema, and rename them to clean `snake_case` names (matching SQL naming conventions).

In [16]:
apps_final_df = apps_clean_df[[
    "app_id", "App", "category_id", "Rating", "Reviews", "Size", "Installs",
    "Type", "Price", "Content Rating", "Last Updated", "Current Ver", "Android Ver"
]].rename(columns={
    "App": "app_name",
    "Rating": "rating",
    "Reviews": "reviews_count",
    "Size": "size",
    "Installs": "installs",
    "Type": "type",
    "Price": "price",
    "Content Rating": "content_rating",
    "Last Updated": "last_updated",
    "Current Ver": "current_version",
    "Android Ver": "android_version"
})

print("Final apps table shape:", apps_final_df.shape)
apps_final_df.head()

Final apps table shape: (10356, 13)


,app_id,app_name,category_id,rating,reviews_count,size,installs,type,price,content_rating,last_updated,current_version,android_version
0,1,Photo Editor & Candy Camera & Grid & ScrapBook,1,4.1,159,19M,10000,Free,0.0,Everyone,"January 7, 2018",1.0.0,4.0.3 and up
1,2,Coloring book moana,1,3.9,967,14M,500000,Free,0.0,Everyone,"January 15, 2018",2.0.0,4.0.3 and up
2,3,"U Launcher Lite – FREE Live Cool Themes, Hide ...",1,4.7,87510,8.7M,5000000,Free,0.0,Everyone,"August 1, 2018",1.2.4,4.0.3 and up
3,4,Sketch - Draw & Paint,1,4.5,215644,25M,50000000,Free,0.0,Teen,"June 8, 2018",Varies with device,4.2 and up
4,5,Pixel Draw - Number Art Coloring Book,1,4.3,967,2.8M,100000,Free,0.0,Everyone,"June 20, 2018",1.1,4.4 and up


In [17]:
# Final check: no nulls, correct types
print(apps_final_df.isnull().sum())
print()
print(apps_final_df.dtypes)

app_id             0
app_name           0
category_id        0
rating             0
reviews_count      0
size               0
installs           0
type               0
price              0
content_rating     0
last_updated       0
current_version    0
android_version    0
dtype: int64

app_id               int64
app_name               str
category_id          int64
rating             float64
reviews_count        int64
size                   str
installs             int64
type                   str
price              float64
content_rating         str
last_updated           str
current_version        str
android_version        str
dtype: object


## 8. Export the cleaned tables

These two CSVs are what we will import into MySQL Workbench. Remember the **import order** from the brief: tables with no foreign keys first!

In [ ]:
# Export -- categories first (no foreign keys), then apps (has category_id as FK)
# encoding="utf-8" is explicit here because some app_name values contain non-ASCII
# characters (Vietnamese, Chinese, Japanese-style brackets, etc.) -- without this,
# the MySQL Import Wizard can silently stop partway through the file.
categories_df.to_csv("categories.csv", index=False, encoding="utf-8")
apps_final_df.to_csv("apps.csv", index=False, encoding="utf-8")

print("Saved categories.csv:", categories_df.shape)
print("Saved apps.csv:", apps_final_df.shape)

---

# Part 2: Cleaning `googleplaystore_user_reviews.csv` (Reviews)

## 1. Load and inspect the raw data

In [19]:
# Load the raw reviews data
reviews_raw_df = pd.read_csv("googleplaystore_user_reviews.csv")

print("Shape:", reviews_raw_df.shape)
reviews_raw_df.head()

Shape: (64295, 5)


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [20]:
reviews_raw_df.isnull().sum()

App                           0
Translated_Review         26868
Sentiment                 26863
Sentiment_Polarity        26863
Sentiment_Subjectivity    26863
dtype: int64

In [21]:
print("Duplicate rows:", reviews_raw_df.duplicated().sum())

Duplicate rows: 33616


**What we found:**
- About 42% of rows have **no review text and no sentiment at all** — these rows carry no usable information for our analysis, so we drop them.
- There are a large number of fully duplicated rows to remove.

## 2. Handle nulls and duplicates

In [22]:
# Work on a copy, never on the raw data
reviews_clean_df = reviews_raw_df.copy()

# Drop rows with no review text -- nothing to analyze there
reviews_clean_df = reviews_clean_df.dropna(subset=["Translated_Review"])

print("Shape after dropping empty reviews:", reviews_clean_df.shape)

Shape after dropping empty reviews: (37427, 5)


In [23]:
# Remove fully duplicated rows
reviews_clean_df = reviews_clean_df.drop_duplicates()

print("Shape after removing duplicates:", reviews_clean_df.shape)
print()
print("Remaining nulls:")
reviews_clean_df.isnull().sum()

Shape after removing duplicates: (29692, 5)

Remaining nulls:


App                       0
Translated_Review         0
Sentiment                 0
Sentiment_Polarity        0
Sentiment_Subjectivity    0
dtype: int64

## 3. Link reviews to apps using `app_id`

Both files share the `App` column (the app name). We use the `app_name → app_id` mapping we already built in Part 1 to attach the correct foreign key to each review — the same idea as `category_to_id`, just applied to a different pair of columns.

In [24]:
# Build the App name -> app_id dictionary from our cleaned apps table
app_name_to_id = dict(zip(apps_final_df["app_name"], apps_final_df["app_id"]))

# Map it onto the reviews table
reviews_clean_df["app_id"] = reviews_clean_df["App"].map(app_name_to_id)

# Some reviews may reference an app that no longer exists in our cleaned apps table
# (for example, if that app was removed as a duplicate). We can't keep a foreign key
# that points to nothing, so we drop those rows.
print("Reviews with no matching app_id:", reviews_clean_df["app_id"].isnull().sum())

Reviews with no matching app_id: 1442


In [25]:
reviews_clean_df = reviews_clean_df.dropna(subset=["app_id"])
reviews_clean_df["app_id"] = reviews_clean_df["app_id"].astype(int)

print("Shape after dropping unmatched reviews:", reviews_clean_df.shape)

Shape after dropping unmatched reviews: (28250, 6)


## 4. Create the `review_id` primary key and build the final table

In [26]:
# Reset index and create review_id
reviews_clean_df = reviews_clean_df.reset_index(drop=True)
reviews_clean_df["review_id"] = reviews_clean_df.index + 1

reviews_final_df = reviews_clean_df[[
    "review_id", "app_id", "Translated_Review", "Sentiment",
    "Sentiment_Polarity", "Sentiment_Subjectivity"
]].rename(columns={
    "Translated_Review": "translated_review",
    "Sentiment": "sentiment",
    "Sentiment_Polarity": "sentiment_polarity",
    "Sentiment_Subjectivity": "sentiment_subjectivity"
})

print("Final reviews table shape:", reviews_final_df.shape)
reviews_final_df.head()

Final reviews table shape: (28250, 6)


,review_id,app_id,translated_review,sentiment,sentiment_polarity,sentiment_subjectivity
0,1,1226,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,2,1226,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,3,1226,Works great especially going grocery store,Positive,0.40,0.875000
3,4,1226,Best idea us,Positive,1.00,0.300000
4,5,1226,Best way,Positive,1.00,0.300000


**Note on line breaks:** the MySQL Workbench Import Wizard cannot reliably parse CSV fields that span multiple lines, even when properly quoted. We flatten `translated_review` to single-line text below before exporting.

In [ ]:
# Flatten any embedded line breaks in translated_review into single spaces.
# Some user reviews contain literal newlines (e.g. multi-paragraph text).
# Even though these are properly quoted in the exported CSV, MySQL Workbench's
# Table Data Import Wizard has a known bug where it miscounts columns on
# multi-line quoted fields, causing an "list index out of range" error.
# Flattening to single-line text avoids this entirely.
reviews_final_df["translated_review"] = (
    reviews_final_df["translated_review"]
    .str.replace(r"[\r\n]+", " ", regex=True)
    .str.strip()
)

print("Line breaks remaining in translated_review:",
      reviews_final_df["translated_review"].str.contains(r"[\r\n]").sum())

# Some reviews also contain literal double-quote characters (e.g. quoting an
# app feature name). Pandas correctly escapes these per the CSV standard by
# doubling them, but MySQL Workbench's Import Wizard has a separate known bug
# where it also fails to parse that doubled-quote escaping correctly, throwing
# the same "list index out of range" error. We replace embedded double quotes
# with single quotes so no CSV escaping is needed at all -- this sidesteps the
# wizard bug entirely.
reviews_final_df["translated_review"] = (
    reviews_final_df["translated_review"]
    .str.replace('"', "'", regex=False)
)

print("Rows with embedded double quotes remaining:",
      reviews_final_df["translated_review"].str.contains('"').sum())

In [27]:
# Final check: no nulls, correct types
print(reviews_final_df.isnull().sum())
print()
print(reviews_final_df.dtypes)

review_id                 0
app_id                    0
translated_review         0
sentiment                 0
sentiment_polarity        0
sentiment_subjectivity    0
dtype: int64

review_id                   int64
app_id                      int64
translated_review             str
sentiment                     str
sentiment_polarity        float64
sentiment_subjectivity    float64
dtype: object


## 5. Export the cleaned table

In [ ]:
# encoding="utf-8" explicit for the same reason as apps.csv above
reviews_final_df.to_csv("reviews.csv", index=False, encoding="utf-8")

print("Saved reviews.csv:", reviews_final_df.shape)

---

## Summary

In this notebook we cleaned the two raw Google Play Store files and produced three tables ready for MySQL:

- **`categories.csv`** — lookup table built from the `Category` column (one value per app → clean one-to-many).
- **`apps.csv`** — the main apps table, with nulls handled, `Reviews`/`Installs`/`Price` converted to proper numeric types, and a `category_id` foreign key.
- **`reviews.csv`** — the reviews table, with empty/duplicate reviews removed and an `app_id` foreign key linking each review back to its app.

**Import order for MySQL Workbench:** `categories.csv` first (no foreign keys), then `apps.csv` (references `categories`), then `reviews.csv` (references `apps`).

**Next steps (later):** write the `CREATE DATABASE` / `CREATE TABLE` SQL script, import these three CSVs via the Table Data Import Wizard, then move on to Day 3 (SQL queries and analysis).